# Cell 1

In [ ]:
# =============================================================================
# Thesis — GeoAI Café Site Selection · Milan
# Phase 2 · Notebook 4 · Feature Engineering (Gold Layer)
# =============================================================================
#
# Purpose:
#   Compute all spatial features per H3 cell centroid and produce the
#   normalized feature table that feeds both the AHP scoring (Notebook 5)
#   and the Random Forest training (Notebook 6).
#
# Feature counts — four distinct totals used consistently throughout:
#   15 GDF columns total   : 13 model features + viable_overlap_ratio + label
#   13 RF model features   : all features passed to Random Forest training
#   10 AHP-scored features : RF features minus 3 RF-only features
#                            (local_cafe_density, office_density,
#                             university_proximity — see RF-only note below)
#    3 RF-only features    : deliberately excluded from AHP on theoretical and
#                            data-quality grounds (see per-feature notes below)
#
# Features computed (13 model features across 4 AHP criteria clusters):
#
#   ACCESSIBILITY (3 features — AHP + RF)
#     1.  metro_count          — metro stations within 400m
#     2.  transit_density      — bus/tram stops within 300m
#     3.  network_centrality   — betweenness centrality of nearest road node
#
#   DEMAND POTENTIAL (2 features — AHP + RF)
#     4.  pop_density          — residents per km² (ISTAT 2021 census tracts)
#     5.  night_light          — VIIRS 2024 average radiance at centroid
#
#   DEMAND POTENTIAL (2 features — RF-ONLY)
#     6.  office_density       — office/commercial POIs within 500m
#                                [RF-ONLY — block-level ISTAT pop_density
#                                 subsumes the office/residential signal;
#                                 excluded from AHP Phase 2]
#     7.  university_proximity — exponential decay from nearest university (0–1)
#                                [RF-ONLY — retained in RF; excluded from AHP
#                                 Phase 2; asymmetry is a documented divergence
#                                 point between the two scoring layers]
#
#   URBAN CONTEXT (4 features — AHP + RF)
#     8.  retail_density       — retail POIs within cell + 1 ring of neighbours
#     9.  tourist_poi_count    — tourism POIs within 500m
#    10.  poi_diversity         — distinct POI categories within cell
#    11.  pedestrian_street     — binary flag: pedestrian street present in cell
#
#   COMPETITION (2 features)
#    12.  local_cafe_density        — cafés within 300m [RF-ONLY — proxies the
#                                     target; excluded from AHP to avoid
#                                     circularity in expert weight elicitation]
#    13.  competitor_saturation_inv_norm — café density / foot traffic proxy
#                                     (inverted) [AHP + RF]
#
#   METADATA COLUMNS (not model features — 2 columns)
#    14.  viable_overlap_ratio — proportion of cell area in viable Urban Atlas zone
#    15.  label                — positive (1) or negative (0) class
#
# All continuous features are normalized to [0, 1] using MinMaxScaler
# fitted exclusively on training cells (no leakage to validation cells).
# The train/validation split (GroupShuffleSplit on H3 res-7 groups) is
# performed BEFORE normalization and saved to Gold for NB6/NB7 to reuse.
#
# Inputs (from Silver folder):
#   - silver_h3_viable.geojson
#   - silver_cafes.geojson
#   - silver_transit_stops.geojson
#   - silver_road_network_nodes.geojson
#   - silver_road_network_edges.geojson
#   - silver_pois.geojson
#   - silver_population.geojson
#   - raw_viirs.tif (from Bronze — raster, not transformed in Silver)
#
# Outputs (Gold folder):
#   - gold_features_raw.geojson        — 15-column GDF, untransformed
#   - gold_features_normalized.geojson — normalized features (model input)
#   - gold_minmax_scaler.pkl           — scaler fitted on training cells only
#   - gold_train_val_idx.pkl           — train/val split indices for NB6/NB7
#   - gold_feature_log.txt
#
# CRS throughout: EPSG:32632 UTM Zone 32N
# =============================================================================

# Cell 2

In [ ]:
!pip install h3 -q
!pip install geopandas -q
!pip install shapely -q
!pip install rasterio -q
!pip install scikit-learn -q
!pip install folium -q
!pip install mapclassify -q
!pip install osmnx -q

import os
import datetime
import warnings

# Scoped warning filters — suppress only known noisy third-party warnings.
# UserWarning and CRS-mismatch warnings from rasterio/pyproj must NOT be
# suppressed so they surface to the user.
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', module='osmnx')
os.environ['OGR_GEOJSON_MAX_OBJ_SIZE'] = '0'

import h3
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
import osmnx as ox
import folium
from sklearn.preprocessing import MinMaxScaler
from google.colab import drive

drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/Thesis'
BRONZE_PATH  = os.path.join(PROJECT_ROOT, 'Bronze')
SILVER_PATH  = os.path.join(PROJECT_ROOT, 'Silver')
GOLD_PATH    = os.path.join(PROJECT_ROOT, 'Gold')
TARGET_CRS   = 'EPSG:32632'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 4.0 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Cell 3

In [ ]:
print("Loading Silver layer inputs...")
print()

# H3 viable grid — the analysis backbone
h3_viable = gpd.read_file(
    os.path.join(SILVER_PATH, 'silver_h3_viable.geojson')
)
h3_viable = h3_viable.to_crs(TARGET_CRS)
print(f"H3 viable grid:      {len(h3_viable)} cells, CRS: {h3_viable.crs}")

# Cafés
cafes = gpd.read_file(
    os.path.join(SILVER_PATH, 'silver_cafes.geojson')
)
cafes = cafes.to_crs(TARGET_CRS)
print(f"Cafés:               {len(cafes)} features")

# Transit stops
transit = gpd.read_file(
    os.path.join(SILVER_PATH, 'silver_transit_stops.geojson')
)
transit = transit.to_crs(TARGET_CRS)
print(f"Transit stops:       {len(transit)} features")

# Road network nodes and edges
road_nodes = gpd.read_file(
    os.path.join(SILVER_PATH, 'silver_road_network_nodes.geojson')
)
road_nodes = road_nodes.to_crs(TARGET_CRS)

road_edges = gpd.read_file(
    os.path.join(SILVER_PATH, 'silver_road_network_edges.geojson')
)
road_edges = road_edges.to_crs(TARGET_CRS)
print(f"Road network:        {len(road_nodes)} nodes, {len(road_edges)} edges")

# General POIs
pois = gpd.read_file(
    os.path.join(SILVER_PATH, 'silver_pois.geojson')
)
pois = pois.to_crs(TARGET_CRS)
print(f"General POIs:        {len(pois)} features")

# silver_population.geojson loaded in Cell 6 (Feature 4 block)

# Extract centroid GeoDataFrame for spatial operations
# The centroid is the measurement anchor for all proximity calculations
centroids = gpd.GeoDataFrame(
    h3_viable[['h3_id', 'label', 'cafe_count',
                'viable_overlap_ratio']].copy(),
    geometry=gpd.points_from_xy(
        h3_viable['centroid_x'],
        h3_viable['centroid_y']
    ),
    crs=TARGET_CRS
)

WGS84_CRS  = 'EPSG:4326'
H3_RESOLUTION = 9

print()
print(f"Centroid GeoDataFrame: {len(centroids)} points")
print("All Silver inputs loaded successfully")

Loading Silver layer inputs...

H3 viable grid:      971 cells, CRS: EPSG:32632
Cafés:               1728 features
Transit stops:       6056 features


/usr/local/lib/python3.12/dist-packages/geopandas/io/file.py:576: UserWarning: Could not parse column 'reversed' as JSON; leaving as string
  return pyogrio.read_dataframe(path_or_bytes, bbox=bbox, **kwargs)


Road network:        86043 nodes, 258724 edges
General POIs:        44552 features

Centroid GeoDataFrame: 971 points
All Silver inputs loaded successfully


# Cell 4

In [ ]:
# This helper function is used by multiple feature computations
# It counts how many points from a reference GeoDataFrame fall
# within a given radius (in metres) of each H3 cell centroid
#
# Using a spatial index (STRtree) for efficiency — without this
# the naive approach would require 392 × n_points distance checks
# which is too slow for larger datasets

from shapely.strtree import STRtree

def count_within_radius(centroids_gdf, reference_gdf, radius_m, label):
    """
    For each centroid, count how many reference points fall within radius_m.
    Returns a Series indexed to match centroids_gdf.

    Parameters
    ----------
    centroids_gdf  : GeoDataFrame of H3 cell centroids
    reference_gdf  : GeoDataFrame of reference points to count
    radius_m       : float, search radius in metres
    label          : str, feature name for progress reporting
    """
    print(f"  Computing {label} (radius={radius_m}m, "
          f"n_reference={len(reference_gdf)})...")

    # Build spatial index on reference points
    ref_geoms = list(reference_gdf.geometry)
    tree      = STRtree(ref_geoms)

    counts = []
    for centroid in centroids_gdf.geometry:
        # Create buffer around centroid
        buffer = centroid.buffer(radius_m)
        # Query spatial index for candidates
        candidates = tree.query(buffer)
        # Count candidates that actually fall within the buffer
        count = sum(
            1 for idx in candidates
            if ref_geoms[idx].within(buffer)
        )
        counts.append(count)

    return pd.Series(counts, index=centroids_gdf.index)

print("Helper function count_within_radius defined")

Helper function count_within_radius defined


# Cell 5

In [ ]:
print("=" * 60)
print("COMPUTING ACCESSIBILITY FEATURES")
print("=" * 60)
print()

# ------------------------------------------------------------------
# Feature 1 — metro_count
# Count of metro station entrances within 400m of cell centroid
# 400m = standard 5-minute walk transit catchment radius
# (urban planning literature standard)
# ------------------------------------------------------------------

metro_stops = transit[transit['transit_type'] == 'metro'].copy()

h3_viable['metro_count'] = count_within_radius(
    centroids, metro_stops, 400, 'metro_count'
)

print(f"  metro_count stats:")
print(f"    Mean:  {h3_viable['metro_count'].mean():.2f}")
print(f"    Max:   {h3_viable['metro_count'].max()}")
print(f"    Cells with metro access: "
      f"{(h3_viable['metro_count'] > 0).sum()}")
print()

# Feature 2 — transit_density
# Count of bus/tram stops within 300m of cell centroid.
# Includes transit_type values 'bus_tram', 'bus_stop', and 'platform':
#   - 'bus_tram'  : stop_position entries tagged public_transport=stop_position
#   - 'bus_stop'  : roadside signs tagged highway=bus_stop (Iteration 5 addition)
#   - 'platform'  : tagged public_transport=platform (Iteration 5 addition)
# Standalone bus_stop entries not co-located with a bus_tram entry within 5m
# retain transit_type == 'bus_stop' after NB2 deduplication and were previously
# excluded, undercounting systematically. metro entries remain separate (Feature 1).
# Shorter radius than metro — bus stops are more numerous and locally distributed.
# ------------------------------------------------------------------

non_metro_stops = transit[
    transit['transit_type'].isin(['bus_tram', 'bus_stop', 'platform'])
].copy()

print(f"  Non-metro stops used for transit_density: {len(non_metro_stops)}")
print(f"    bus_tram:  {(non_metro_stops['transit_type'] == 'bus_tram').sum()}")
print(f"    bus_stop:  {(non_metro_stops['transit_type'] == 'bus_stop').sum()}")
print(f"    platform:  {(non_metro_stops['transit_type'] == 'platform').sum()}")

h3_viable['transit_density'] = count_within_radius(
    centroids, non_metro_stops, 300, 'transit_density'
)

print(f"  transit_density stats:")
print(f"    Mean:  {h3_viable['transit_density'].mean():.2f}")
print(f"    Max:   {h3_viable['transit_density'].max()}")
print()

# ------------------------------------------------------------------
# Feature 3 — network_centrality
# Betweenness centrality of the nearest road node to each centroid.
#
# Approximation note: k=500 samples ~0.44% of 114k nodes when using
# approximate betweenness. To guarantee reproducibility for a thesis
# deliverable, we validate rank stability by running k=500 and k=1000
# and reporting the Spearman rank correlation between the two vectors.
# seed=42 is fixed so every run produces identical results.
#
# WARNING: Heavy processing block. Expected runtime: ~40 minutes.
# ------------------------------------------------------------------

print("  Computing network_centrality...")
print("  Building road graph for betweenness centrality...")

import networkx as nx
from scipy.stats import spearmanr as _spearmanr

# --- Prepare nodes and edges for manual graph construction ---
nodes_centrality = road_nodes.copy()
if nodes_centrality.crs is None:
    nodes_centrality.set_crs(TARGET_CRS, inplace=True)

edges_centrality = road_edges.copy()
if edges_centrality.crs is None:
    edges_centrality.set_crs(TARGET_CRS, inplace=True)

edges_centrality['length'] = edges_centrality.geometry.length

# --- Build directed graph ---
print("  Creating directed graph from nodes and edges (DiGraph)...")
G = nx.DiGraph()

node_ids = (nodes_centrality['osmid'].values
            if 'osmid' in nodes_centrality.columns
            else nodes_centrality.index.values)
xs = nodes_centrality.geometry.x.values
ys = nodes_centrality.geometry.y.values
G.add_nodes_from(zip(node_ids, [{'x': x, 'y': y} for x, y in zip(xs, ys)]))

print(f"  Added {G.number_of_nodes()} nodes")

if 'u' not in edges_centrality.columns or 'v' not in edges_centrality.columns:
    raise ValueError("Edges must have 'u' and 'v' columns for source and target nodes")

if 'key' not in edges_centrality.columns:
    edges_centrality['key'] = 0

G.add_edges_from(
    [(r.u, r.v, {'length': r.length})
     for r in edges_centrality[['u', 'v', 'length']].itertuples()]
)
print(f"  Added {G.number_of_edges()} edges (directed)")

# --- Compute approximate betweenness with k=500, seed=42 ---
k_samples = min(500, G.number_of_nodes())
print(f"  Computing betweenness centrality (k={k_samples}, seed=42)...")
centrality_dict_500 = nx.betweenness_centrality(
    G, k=k_samples, weight='length', normalized=True, seed=42
)

# --- Rank-stability check: compare k=500 vs k=1000 ---
# The seed=42 fixes random-sample selection given a fixed graph, but
# add_edges_from insertion order (derived from gpd.clip() row order in NB2)
# affects node ordering for the k-sample and is not guaranteed stable if the
# Silver layer is regenerated. The rank-stability check catches non-reproducibility
# empirically; ρ < 0.99 is converted to a hard stop to prevent silent drift.
k_check = min(1000, G.number_of_nodes())
if k_check > k_samples:
    print(f"  Running rank-stability check (k={k_check}, seed=42)...")
    centrality_dict_1000 = nx.betweenness_centrality(
        G, k=k_check, weight='length', normalized=True, seed=42
    )
    _node_ids_check = list(centrality_dict_500.keys())
    vec_500   = [centrality_dict_500[n]  for n in _node_ids_check]
    vec_1000  = [centrality_dict_1000[n] for n in _node_ids_check]
    rho, _    = _spearmanr(vec_500, vec_1000)
    print(f"  Rank-stability Spearman ρ (k=500 vs k=1000): {rho:.4f}")
    if rho >= 0.99:
        print(f"  [PASS] Rank correlation ≥ 0.99 — approximation is rank-stable.")
    else:
        assert rho >= 0.99, (
            f"Betweenness rank-stability check FAILED (ρ={rho:.4f}) — "
            f"re-run may be non-reproducible. Investigate Silver layer row order "
            f"or increase k. See NB4 Feature 3 documentation."
        )
else:
    print(f"  Rank-stability check skipped — graph has ≤ 1000 nodes "
          f"(exact computation already in effect).")

# Use k=500 result as the canonical centrality (seed fixed → deterministic)
centrality_dict = centrality_dict_500

# --- Map centrality scores back to nodes GeoDataFrame ---
road_nodes_centrality = nodes_centrality.copy()
if 'osmid' in road_nodes_centrality.columns:
    road_nodes_centrality['centrality'] = (
        road_nodes_centrality['osmid'].map(centrality_dict).fillna(0)
    )
else:
    road_nodes_centrality['centrality'] = (
        road_nodes_centrality.index.map(centrality_dict).fillna(0)
    )

# Ensure contiguous index before STRtree build
road_nodes_centrality = road_nodes_centrality.reset_index(drop=True)

# Build spatial index on road nodes
road_node_geoms = list(road_nodes_centrality.geometry)
road_tree = STRtree(road_node_geoms)

# Assign centrality of nearest road node to each H3 centroid
centrality_values = []
for centroid in centroids.geometry:
    nearest_idx = road_tree.nearest(centroid)
    centrality  = road_nodes_centrality['centrality'].iloc[nearest_idx]
    centrality_values.append(centrality)

h3_viable['network_centrality'] = centrality_values

print(f"  network_centrality stats:")
print(f"    Mean:  {h3_viable['network_centrality'].mean():.6f}")
print(f"    Max:   {h3_viable['network_centrality'].max():.6f}")
print(f"  Approximation: k={k_samples} samples, seed=42 (deterministic)")
print(f"  Graph type: DiGraph (directed — respects one-way streets)")
print()
print("Accessibility features complete")

COMPUTING ACCESSIBILITY FEATURES

  Computing metro_count (radius=400m, n_reference=115)...
  metro_count stats:
    Mean:  0.50
    Max:   4
    Cells with metro access: 361

  Non-metro stops used for transit_density: 5941
    bus_tram:  3242
    bus_stop:  0
    platform:  2699
  Computing transit_density (radius=300m, n_reference=5941)...
  transit_density stats:
    Mean:  13.34
    Max:   54

  Computing network_centrality...
  Building road graph for betweenness centrality...
  Creating directed graph from nodes and edges (DiGraph)...
  Added 86043 nodes
  Added 256266 edges (directed)
  Computing betweenness centrality (k=500, seed=42)...
  Running rank-stability check (k=1000, seed=42)...
  Rank-stability Spearman ρ (k=500 vs k=1000): 0.9909
  [PASS] Rank correlation ≥ 0.99 — approximation is rank-stable.
  network_centrality stats:
    Mean:  0.001586
    Max:   0.035649
  Approximation: k=500 samples, seed=42 (deterministic)
  Graph type: DiGraph (directed — respects one-way

# Cell 6

In [ ]:
print("=" * 60)
print("COMPUTING DEMAND POTENTIAL FEATURES")
print("=" * 60)
print()

# ------------------------------------------------------------------
# Feature 4 — pop_density
# Population density in residents per km²
#
# silver_population.geojson is the canonical clean layer produced by
# NB2. NB2 applies the ISTAT-vs-fallback branch and writes a
# pop_density_per_km2 column in both cases. NB4 unconditionally
# consumes that Silver output — Bronze is not re-read here.
# ------------------------------------------------------------------

print("  Loading population density from silver_population.geojson...")
population = gpd.read_file(
    os.path.join(SILVER_PATH, 'silver_population.geojson')
)
population = population.to_crs(TARGET_CRS)
print(f"  Population layer: {len(population)} features")

if 'pop_density_per_km2' not in population.columns:
    raise KeyError(
        "Expected 'pop_density_per_km2' column not found in "
        "silver_population.geojson. Re-run Notebook 2 to regenerate."
    )

# Two-pass spatial join: within (unambiguous) then intersects (boundary residuals, nearest-tract tiebreak)
pop_cols = population[['pop_density_per_km2', 'geometry']].copy()

# ── Pass 1: within ────────────────────────────────────────────────────────────
joined_within = gpd.sjoin(
    centroids[['h3_id', 'geometry']],
    pop_cols,
    how='left',
    predicate='within'
)
# Drop duplicates that can arise if a centroid is within multiple overlapping
# clipped polygons (geometry artefact); keep first match (arbitrary but
# 'within' should uniquely assign for non-overlapping tracts).
joined_within = joined_within.drop_duplicates(subset='h3_id')

matched_ids   = joined_within.loc[
    joined_within['pop_density_per_km2'].notna(), 'h3_id'
]
unmatched_mask = ~centroids['h3_id'].isin(matched_ids)
centroids_unmatched = centroids[unmatched_mask].copy()

print(f"  Pass 1 (within):      {matched_ids.nunique()} centroids matched")
print(f"  Residual unmatched:   {len(centroids_unmatched)} centroids → Pass 2")

# ── Pass 2: intersects on residual, nearest‑tract tiebreak ─────────────────
if len(centroids_unmatched) > 0:
    # Reset index so index_right from sjoin matches dict keys exactly
    population_reset = population.reset_index(drop=True)
    _idx_to_tract_centroid = {
        idx: geom.centroid
        for idx, geom in zip(population_reset.index, population_reset.geometry)
    }

    # Re-run Pass 2 sjoin using the reset-index GeoDataFrame so that
    # index_right values are guaranteed 0…N-1 and match the dict keys.
    pop_cols_reset = population_reset[['pop_density_per_km2', 'geometry']].copy()
    joined_intersects = gpd.sjoin(
        centroids_unmatched[['h3_id', 'geometry']],
        pop_cols_reset,
        how='left',
        predicate='intersects'
    )

    # Guard: assert every index_right value present in joined_intersects
    # is a valid key in _idx_to_tract_centroid before nearest-distance lookup.
    ir_values = joined_intersects['index_right'].dropna().astype(int)
    missing_keys = set(ir_values) - set(_idx_to_tract_centroid.keys())
    assert not missing_keys, (
        f"nearest_density: {len(missing_keys)} index_right value(s) not found "
        f"in _idx_to_tract_centroid — population index mismatch. "
        f"Missing keys (sample): {sorted(missing_keys)[:10]}"
    )

    if joined_intersects['pop_density_per_km2'].notna().any():
        _h3id_to_geom = dict(
            zip(centroids_unmatched['h3_id'], centroids_unmatched['geometry'])
        )

        def nearest_density(group):
            h3_geom = _h3id_to_geom[group.name]
            dists = group['index_right'].map(
                lambda idx: h3_geom.distance(
                    _idx_to_tract_centroid.get(int(idx), h3_geom)
                )
            )
            return group.loc[dists.idxmin()]

        joined_intersects = (
            joined_intersects
            .groupby('h3_id', group_keys=False)
            .apply(nearest_density)
            .reset_index(drop=True)
        )
    print(f"  Pass 2 (intersects):  "
          f"{joined_intersects['pop_density_per_km2'].notna().sum()} residual centroids matched")
else:
    joined_intersects = gpd.GeoDataFrame(
        columns=joined_within.columns, crs=centroids.crs
    )
    print(f"  Pass 2 (intersects):  skipped — no unmatched centroids")

# ── Merge passes and map onto h3_viable ──────────────────────────────────────
density_within     = joined_within.set_index('h3_id')['pop_density_per_km2']
density_intersects = (
    joined_intersects.set_index('h3_id')['pop_density_per_km2']
    if len(joined_intersects) > 0
    else pd.Series(dtype=float)
)
pop_density_map = density_within.combine_first(density_intersects)

h3_viable['pop_density'] = h3_viable['h3_id'].map(pop_density_map).fillna(0)

print(f"  pop_density source: silver_population.geojson "
      f"(spatial resolution determined by NB2 source branch)")
print(f"\n  pop_density stats:")
print(f"    Mean:  {h3_viable['pop_density'].mean():.1f} residents/km²")
print(f"    Max:   {h3_viable['pop_density'].max():.1f} residents/km²")
print()

# ------------------------------------------------------------------
# Feature 5 — office_density
# Count of office/commercial POIs within 500m of centroid
# ------------------------------------------------------------------

office_mask_base = (
    (pois.get('building', pd.Series(dtype=str)).str.lower() == 'office') |
    (pois.get('landuse', pd.Series(dtype=str)).str.lower().isin(['commercial', 'retail'])) |
    (pois.get('office', pd.Series(dtype=str)).notna())
) & ~(pois.get('amenity', pd.Series(dtype=str)).str.lower() == 'cafe')

office_mask = office_mask_base
office_pois = pois[office_mask].copy()
print(f"  Office/commercial POIs identified (excluding cafes): {len(office_pois)}")

h3_viable['office_density'] = count_within_radius(
    centroids, office_pois, 500, 'office_density'
)

print(f"  office_density stats:")
print(f"    Mean:  {h3_viable['office_density'].mean():.2f}")
print(f"    Max:   {h3_viable['office_density'].max()}")
print()

# ------------------------------------------------------------------
# Feature 6 — university_proximity
# Exponential decay score from each centroid to the nearest university.
#
# Use raw geometry — distance() returns 0 for points inside a polygon (correct for campus centroids)
# ------------------------------------------------------------------

print("  Computing university_proximity...")

university_mask = (
    pois.get('amenity', pd.Series(dtype=str)).str.lower() == 'university'
)
universities = pois[university_mask].copy()

print(f"  Universities found: {len(universities)}")

if len(universities) > 0:
    # Use raw geometry — Shapely distance() handles Polygon interior correctly
    uni_geometries = universities.geometry
    uni_geometries = uni_geometries[~uni_geometries.is_empty]

    uni_geoms = list(uni_geometries)
    uni_tree  = STRtree(uni_geoms)

    distances = []
    for centroid in centroids.geometry:
        if len(uni_geoms) == 0:
            distances.append(np.nan)
            continue

        nearest_idx = uni_tree.nearest(centroid)

        if nearest_idx is None:
            distances.append(np.nan)
            continue

        nearest_geom = uni_geoms[nearest_idx]
        # distance() returns 0 for points inside a Polygon — correct behaviour
        dist = centroid.distance(nearest_geom)
        distances.append(dist)

    h3_viable['university_proximity'] = np.exp(-np.array(distances) / 1000.0)

else:
    print("  No universities found in study area — checking OSM directly...")
    try:
        _study_area_fallback = gpd.read_file(
            os.path.join(SILVER_PATH, 'silver_study_area.geojson')
        )
        study_polygon_wgs84 = _study_area_fallback.to_crs(WGS84_CRS).dissolve().geometry.iloc[0]

        unis_osm = ox.features_from_polygon(
            study_polygon_wgs84,
            tags={'amenity': 'university'}
        )
        unis_osm = unis_osm.to_crs(TARGET_CRS)
        print(f"  Universities retrieved from OSM: {len(unis_osm)}")

        if len(unis_osm) > 0:
            # Use raw geometry (not .boundary) — same fix as primary path above
            uni_geometries = unis_osm.geometry
            uni_geometries = uni_geometries[~uni_geometries.is_empty]

            uni_geoms = list(uni_geometries)
            uni_tree  = STRtree(uni_geoms)

            distances = []
            for centroid in centroids.geometry:
                if len(uni_geoms) == 0:
                    distances.append(np.nan)
                    continue

                nearest_idx = uni_tree.nearest(centroid)
                if nearest_idx is None:
                    distances.append(np.nan)
                    continue

                nearest_geom = uni_geoms[nearest_idx]
                dist = centroid.distance(nearest_geom)
                distances.append(dist)

            h3_viable['university_proximity'] = np.exp(-np.array(distances) / 1000.0)
        else:
            print("  WARNING: university_proximity set to 0.0 for all cells — no universities found via POI layer or OSM fallback.")
            h3_viable['university_proximity'] = 0.0
    except Exception as e:
        print(f"  OSM query failed: {e}")
        h3_viable['university_proximity'] = 0.0

print(f"\n  university_proximity stats (0 to 1 scale):")
print(f"    Mean:  {h3_viable['university_proximity'].mean():.4f}")
print(f"    Max:   {h3_viable['university_proximity'].max():.4f}")
print()

# ------------------------------------------------------------------
# Feature 7 — night_light
# VIIRS 2024 average radiance value at each cell centroid
# ------------------------------------------------------------------

print("  Computing night_light from VIIRS raster...")

VIIRS_PATH = os.path.join(BRONZE_PATH, 'raw_viirs.tif')

if os.path.exists(VIIRS_PATH):
    with rasterio.open(VIIRS_PATH) as src:
        print(f"  VIIRS CRS: {src.crs}")

        centroids_viirs = centroids.to_crs(src.crs)

        print("  Sampling raster values (optimized)...")
        coords = [(geom.x, geom.y) for geom in centroids_viirs.geometry]

        night_light_values = []

        for val in src.sample(coords):
            v = float(val[0])
            if v < 0:
                v = 0.0
            night_light_values.append(v)

        h3_viable['night_light'] = night_light_values
        print(f"  VIIRS data source: 2024 annual average")

else:
    print("  MISSING: raw_viirs.tif not found in Bronze folder")
    print("  night_light set to 0 for all cells")
    h3_viable['night_light'] = 0.0

print(f"\n  night_light stats:")
print(f"    Mean:  {h3_viable['night_light'].mean():.4f}")
print(f"    Max:   {h3_viable['night_light'].max():.4f}")
print()
print("Demand potential features complete")

COMPUTING DEMAND POTENTIAL FEATURES

  Loading population density from silver_population.geojson...
  Population layer: 7296 features
  Pass 1 (within):      971 centroids matched
  Residual unmatched:   0 centroids → Pass 2
  Pass 2 (intersects):  skipped — no unmatched centroids
  pop_density source: silver_population.geojson (spatial resolution determined by NB2 source branch)

  pop_density stats:
    Mean:  12377.8 residents/km²
    Max:   92053.9 residents/km²

  Office/commercial POIs identified (excluding cafes): 2862
  Computing office_density (radius=500m, n_reference=2862)...
  office_density stats:
    Mean:  17.85
    Max:   142

  Computing university_proximity...
  Universities found: 47

  university_proximity stats (0 to 1 scale):
    Mean:  0.3522
    Max:   0.9782

  Computing night_light from VIIRS raster...
  VIIRS CRS: EPSG:4326
  Sampling raster values (optimized)...
  VIIRS data source: 2024 annual average

  night_light stats:
    Mean:  53.0826
    Max:   137.

# Cell 7

In [ ]:
print("=" * 60)
print("COMPUTING URBAN CONTEXT FEATURES")
print("=" * 60)
print()

# ------------------------------------------------------------------
# Feature 8 — retail_density
# Count of retail POIs within the cell and its immediate H3
# ring of neighbours (resolution 9 ring 1)
# Using the H3 neighbour ring rather than a fixed radius ensures
# geometrically consistent neighbourhood definitions — all six
# neighbours are equidistant from the cell center
# ------------------------------------------------------------------

print("  Computing retail_density using H3 neighbour ring...")

# Filter POIs to retail categories
retail_mask = (
    (pois.get('shop', pd.Series(dtype=str)).notna()) |
    (pois.get('amenity', pd.Series(dtype=str)).str.lower().isin(['marketplace', 'supermarket']))
) & ~(pois.get('amenity', pd.Series(dtype=str)).str.lower() == 'cafe')
retail_pois = pois[retail_mask].copy()
print(f"  Retail POIs identified: {len(retail_pois)}")

# Reproject retail POIs to WGS84 for H3 cell assignment
retail_wgs84 = retail_pois.to_crs(WGS84_CRS)

# Assign each retail POI to its H3 cell
def assign_h3(geom, res=H3_RESOLUTION):
    try:
        return h3.latlng_to_cell(geom.y, geom.x, res)
    except Exception:
        return None

retail_wgs84['h3_id'] = retail_wgs84.geometry.apply(assign_h3)
retail_wgs84 = retail_wgs84.dropna(subset=['h3_id'])

# Count retail POIs per cell including immediate ring of neighbours
retail_per_cell = retail_wgs84.groupby('h3_id').size().to_dict()

def retail_density_with_ring(h3_id):
    try:
        # Get the cell itself and its 6 immediate neighbours
        neighbours = h3.grid_disk(h3_id, 1)
        total = sum(retail_per_cell.get(n, 0) for n in neighbours)
        return total
    except Exception:
        return 0

h3_viable['retail_density'] = h3_viable['h3_id'].apply(
    retail_density_with_ring
)

print(f"  retail_density stats:")
print(f"    Mean:  {h3_viable['retail_density'].mean():.2f}")
print(f"    Max:   {h3_viable['retail_density'].max()}")
print()

# ------------------------------------------------------------------
# Feature 9 — tourist_poi_count
# Count of tourism-tagged POIs within 500m of centroid
# ------------------------------------------------------------------

tourist_mask = (
    pois.get('tourism', pd.Series(dtype=str)).notna()
) | (
    pois.get('amenity', pd.Series(dtype=str)).str.lower().isin(
        ['theatre', 'cinema', 'arts_centre', 'museum']
    )
)
tourist_pois = pois[tourist_mask].copy()
print(f"  Tourism POIs identified: {len(tourist_pois)}")

h3_viable['tourist_poi_count'] = count_within_radius(
    centroids, tourist_pois, 500, 'tourist_poi_count'
)

print(f"  tourist_poi_count stats:")
print(f"    Mean:  {h3_viable['tourist_poi_count'].mean():.2f}")
print(f"    Max:   {h3_viable['tourist_poi_count'].max()}")
print()

# ------------------------------------------------------------------
# Feature 10 — poi_diversity
# Count of distinct OSM tag categories present within the cell only
# (no ring-1 neighbourhood — cell boundary only)
# Higher diversity = more mixed-use environment
# Motivated by Han et al. 2024 co-location pattern literature
#
# NOTE — deliberate scope asymmetry: retail_density uses a ring-1
# H3 neighbourhood (~7 cells) to capture nearby retail context, while
# poi_diversity uses the cell boundary only. This is intentional:
# diversity is a property of the local micro-environment inside the
# cell; diluting it across 7 cells would make all interior cells in a
# dense district look equally diverse and lose the within-district
# signal. retail_density by contrast benefits from the neighbourhood
# because a cell on the boundary of a retail district should not score
# zero retail simply because most shops are in the adjacent cell.
# This asymmetry is documented as a methodological choice; see thesis
# footnote on feature scope decisions.
# ------------------------------------------------------------------

print("  Computing poi_diversity...")

# Assign all POIs to H3 cells
pois_wgs84 = pois.to_crs(WGS84_CRS).copy()
pois_wgs84['h3_id'] = pois_wgs84.geometry.apply(assign_h3)
pois_wgs84 = pois_wgs84.dropna(subset=['h3_id'])

# Determine primary category for each POI
def get_primary_category(row):
    for col in ['amenity', 'shop', 'tourism', 'building',
                'landuse', 'office']:
        if col in row and pd.notna(row[col]) and str(row[col]) != 'nan':
            return f"{col}={row[col]}"
    return None

pois_wgs84['primary_category'] = pois_wgs84.apply(
    get_primary_category, axis=1
)
pois_wgs84 = pois_wgs84.dropna(subset=['primary_category'])

# Count distinct categories per H3 cell (cell boundary only — no ring)
diversity_per_cell = (
    pois_wgs84.groupby('h3_id')['primary_category']
    .nunique()
    .to_dict()
)

h3_viable['poi_diversity'] = h3_viable['h3_id'].map(
    diversity_per_cell
).fillna(0).astype(int)

print(f"  poi_diversity stats:")
print(f"    Mean:  {h3_viable['poi_diversity'].mean():.2f}")
print(f"    Max:   {h3_viable['poi_diversity'].max()}")
print()
print("Urban context features complete")

COMPUTING URBAN CONTEXT FEATURES

  Computing retail_density using H3 neighbour ring...
  Retail POIs identified: 9095
  retail_density stats:
    Mean:  59.46
    Max:   520

  Tourism POIs identified: 1773
  Computing tourist_poi_count (radius=500m, n_reference=1773)...
  tourist_poi_count stats:
    Mean:  11.57
    Max:   137

  Computing poi_diversity...
  poi_diversity stats:
    Mean:  17.68
    Max:   75

Urban context features complete


# Cell 8

In [ ]:
print("=" * 60)
print("COMPUTING COMPETITION FEATURES")
print("=" * 60)
print()

# ------------------------------------------------------------------
# Feature 11 — local_cafe_density
# Count of existing cafés within 300m of centroid
# Distinct from competitor_saturation — this is the raw count
# NOTE: local_cafe_density counts cafés within 300m of the H3 centroid,
# which geometrically overlaps with the label definition (≥2 cafés per cell).
# This feature is intentionally retained as a proxy for agglomeration
# externalities, not as a label predictor. It is excluded from AHP and flagged
# RF-only. See NB6 sensitivity ablation (retrain without local_cafe_density)
# for AUC robustness check.
# ------------------------------------------------------------------

h3_viable['local_cafe_density'] = count_within_radius(
    centroids, cafes, 300, 'local_cafe_density'
)

print(f"  local_cafe_density stats:")
print(f"    Mean:  {h3_viable['local_cafe_density'].mean():.2f}")
print(f"    Max:   {h3_viable['local_cafe_density'].max()}")
print()

# ------------------------------------------------------------------
# Feature 12 — competitor_saturation
# Café density relative to foot traffic proxy
# Formula: local_cafe_density / (transit_density + retail_count_300m + 1)
# The +1 prevents division by zero in cells with no transit or retail
#
# retail_count_300m uses 300 m radius to match transit_density geometry in the denominator
# ------------------------------------------------------------------

print("  Computing retail_count_300m for competitor_saturation denominator...")

retail_count_300m = count_within_radius(
    centroids, retail_pois, 300, 'retail_count_300m'
)

print("  Computing competitor_saturation...")

foot_traffic_proxy = (
    h3_viable['transit_density'] +
    retail_count_300m +
    1  # prevent division by zero
)

h3_viable['competitor_saturation'] = (
    h3_viable['local_cafe_density'] / foot_traffic_proxy
)

print(f"  competitor_saturation denominator: transit_density + retail_count_300m + 1")
print(f"  (retail_count_300m uses 300 m radius, matching transit_density geometry)")
print(f"  competitor_saturation stats:")
print(f"    Mean:  {h3_viable['competitor_saturation'].mean():.4f}")
print(f"    Max:   {h3_viable['competitor_saturation'].max():.4f}")
print()

# ------------------------------------------------------------------
# Feature 13 — pedestrian_street
# Binary flag: 1 if any pedestrian street exists within the cell
# Pedestrian zones generate high foot traffic and are prime
# commercial locations
# ------------------------------------------------------------------

print("  Computing pedestrian_street flag...")

highway_col = road_edges.get('highway', pd.Series(dtype=str)).astype(str).str.lower()

pedestrian_mask = highway_col.str.contains('pedestrian|living_street', na=False)
pedestrian_edges = road_edges[pedestrian_mask].copy()

print(f"  Pedestrian road segments found: {len(pedestrian_edges)}")

if len(pedestrian_edges) > 0:
    # Assign each pedestrian edge midpoint to its H3 cell
    ped_wgs84 = pedestrian_edges.to_crs(WGS84_CRS).copy()

    # representative_point() guarantees the point is inside the geometry
    ped_wgs84['rep_point'] = ped_wgs84.geometry.representative_point()
    ped_wgs84 = ped_wgs84.set_geometry('rep_point')

    # Suppress potential warnings about copying geographically
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        ped_wgs84['h3_id'] = ped_wgs84.geometry.apply(assign_h3)

    ped_wgs84 = ped_wgs84.dropna(subset=['h3_id'])

    pedestrian_cells = set(ped_wgs84['h3_id'].unique())

    h3_viable['pedestrian_street'] = h3_viable['h3_id'].isin(
        pedestrian_cells
    ).astype(int)
else:
    h3_viable['pedestrian_street'] = 0

print(f"  Cells with pedestrian streets: "
      f"{h3_viable['pedestrian_street'].sum()}")
print()
print("Competition features complete")

COMPUTING COMPETITION FEATURES

  Computing local_cafe_density (radius=300m, n_reference=1728)...
  local_cafe_density stats:
    Mean:  4.44
    Max:   34

  Computing retail_count_300m for competitor_saturation denominator...
  Computing retail_count_300m (radius=300m, n_reference=9095)...
  Computing competitor_saturation...
  competitor_saturation denominator: transit_density + retail_count_300m + 1
  (retail_count_300m uses 300 m radius, matching transit_density geometry)
  competitor_saturation stats:
    Mean:  0.1037
    Max:   1.0000

  Computing pedestrian_street flag...
  Pedestrian road segments found: 4260
  Cells with pedestrian streets: 205

Competition features complete


# Cell 9

In [ ]:
# =============================================================================
# Save truly raw (pre-transform) features, then normalize
# =============================================================================

FEATURE_COLS_ORDERED = [
    'h3_id', 'label', 'cafe_count', 'viable_overlap_ratio',
    'metro_count', 'transit_density', 'network_centrality',
    'pop_density', 'office_density', 'university_proximity', 'night_light',
    'retail_density', 'tourist_poi_count', 'poi_diversity',
    'local_cafe_density', 'competitor_saturation', 'pedestrian_street',
    'geometry', 'centroid_x', 'centroid_y'
]

cols_present = [c for c in FEATURE_COLS_ORDERED if c in h3_viable.columns]

# ── Step 1: Save the genuinely untransformed values ───────────────────────────
features_raw = h3_viable[cols_present].copy()           # untouched source
features_raw_wgs84 = features_raw.to_crs(WGS84_CRS)
raw_path = os.path.join(GOLD_PATH, 'gold_features_raw.geojson')
features_raw_wgs84.to_file(raw_path, driver='GeoJSON')
print(f"Raw (pre-transform) features saved: gold_features_raw.geojson")
print(f"  Shape: {features_raw.shape}")
print()

# Print summary on raw values for the log
numeric_features = [
    'metro_count', 'transit_density', 'network_centrality',
    'pop_density', 'office_density', 'university_proximity', 'night_light',
    'retail_density', 'tourist_poi_count', 'poi_diversity',
    'local_cafe_density', 'competitor_saturation', 'pedestrian_street'
]
print("Feature summary (raw, pre-transform values):")
print(features_raw[numeric_features].describe().round(4).to_string())

Raw (pre-transform) features saved: gold_features_raw.geojson
  Shape: (971, 20)

Feature summary (raw, pre-transform values):
       metro_count  transit_density  network_centrality  pop_density  office_density  university_proximity  night_light  retail_density  tourist_poi_count  poi_diversity  local_cafe_density  competitor_saturation  pedestrian_street
count     971.0000         971.0000            971.0000     971.0000        971.0000              971.0000     971.0000        971.0000           971.0000       971.0000            971.0000               971.0000           971.0000
mean        0.5005          13.3440              0.0016   12377.7627         17.8527                0.3522      53.0826         59.4645            11.5747        17.6818              4.4398                 0.1037             0.2111
std         0.7508           8.1282              0.0039   12529.1937         20.0251                0.2239      16.6502         76.7632            19.2423        14.7048        

# Cell 10

In [ ]:
# Cell 10 — Log-transform + MinMaxScaler normalization (on a copy)
#
# NOTE: The train/validation split is performed here, BEFORE normalization,
# so that MinMaxScaler.fit() never sees validation cell values.
# The val_idx produced here is saved to Gold and consumed by NB6 and NB7.
# NB6 must load this split rather than recompute it, to guarantee consistency.
#
# Spatial separation note: GroupShuffleSplit keyed on H3 resolution-7 parent
# cells provides partial spatial separation. Resolution-9 cells on opposite
# sides of a resolution-7 boundary can still be within ~350 m of each other
# and may share correlated feature values. This is documented as a caveat in
# the methodology chapter; H3 resolution-6 grouping is identified as a future
# strengthening approach.

import h3 as h3_lib
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import MinMaxScaler
import pickle

# Contiguous index required for unambiguous positional indexing by GroupShuffleSplit
features_raw = features_raw.reset_index(drop=True)

print("Step 1 — Computing train/validation split (before normalization)...")

# Partial spatial separation via H3 resolution-7 grouping (~5.1 km² hexagons)
# Adjacent cells across group boundaries may still share correlated features
# within ~350 m — this is not full spatial independence (see thesis caveat).
groups = np.array([h3_lib.cell_to_parent(c, 7) for c in features_raw['h3_id']])

# 75/25 split matching the NB6 methodology spec
gss = GroupShuffleSplit(n_splits=1, train_size=0.75, random_state=42)
train_idx, val_idx = next(
    gss.split(np.zeros(len(features_raw)), features_raw['label'].values, groups)
)

print(f"  Train cells: {len(train_idx)}  |  Validation cells: {len(val_idx)}")

# Save split indices so NB6 and NB7 use the *identical* partition
split_path = os.path.join(GOLD_PATH, 'gold_train_val_idx.pkl')
with open(split_path, 'wb') as f:
    pickle.dump({'train_idx': train_idx, 'val_idx': val_idx}, f)
print(f"  Split indices saved: gold_train_val_idx.pkl")
print()

# ── Step 2: Apply log1p to skewed features (on a COPY of features_raw) ────────
print("Step 2 — Applying log1p transform to skewed features (on copy)...")

features_to_norm = features_raw.copy()   # features_raw remains untransformed

# pop_density and night_light included: right-skewed with high outliers
skewed_features = [
    'metro_count', 'transit_density', 'office_density',
    'retail_density', 'tourist_poi_count', 'poi_diversity', 'local_cafe_density',
    'pop_density', 'night_light',
]
print(f"  log1p applied to: {skewed_features}")
print(f"  Rationale for pop_density + night_light: right-skewed distributions "
      f"with high outliers; log1p prevents outlier leverage in MinMaxScaler.")

for col in skewed_features:
    if col in features_to_norm.columns:
        features_to_norm[col] = np.log1p(features_to_norm[col])

# ── Step 3: Invert competitor_saturation using training-set max ONLY ──────────
# Inversion reference computed from training cells only — high saturation = bad opportunity
print("Step 3 — Inverting competitor_saturation (training-set max, no leakage)...")

sat_max_train = features_to_norm['competitor_saturation'].iloc[train_idx].max()
features_to_norm['competitor_saturation_inverted'] = (
    sat_max_train - features_to_norm['competitor_saturation']
)

# Save sat_max_train to Gold for reproducibility in inference
sat_max_path = os.path.join(GOLD_PATH, 'gold_sat_max_train.pkl')
with open(sat_max_path, 'wb') as f:
    pickle.dump(sat_max_train, f)
print(f"  sat_max_train = {sat_max_train:.6f} (saved to gold_sat_max_train.pkl)")

continuous_cols = [
    'metro_count', 'transit_density', 'network_centrality',
    'pop_density', 'office_density', 'university_proximity',
    'night_light', 'retail_density', 'tourist_poi_count',
    'poi_diversity', 'local_cafe_density', 'competitor_saturation_inverted'
]

# ── Step 4: Fit scaler ONLY on training cells, transform all cells ─────────────
print("Step 4 — Fitting MinMaxScaler on training cells only (no leakage)...")

X_continuous = features_to_norm[continuous_cols].fillna(0).values
X_train_cont = X_continuous[train_idx]     # scaler sees only training cells

scaler = MinMaxScaler()
scaler.fit(X_train_cont)                         # fit on train only
X_normalized = scaler.transform(X_continuous)   # transform all (train + val)

# Build normalized column names; rename inverted saturation to canonical name
norm_col_names = [f"{c}_norm" for c in continuous_cols]
normalized_df = pd.DataFrame(
    X_normalized,
    columns=norm_col_names,
    index=features_to_norm.index
)
# Rename to canonical column name
normalized_df = normalized_df.rename(columns={
    'competitor_saturation_inverted_norm': 'competitor_saturation_inv_norm'
})

# ── Step 5: Assemble normalized GeoDataFrame ───────────────────────────────────
# pedestrian_street is a binary flag (0/1) and is intentionally passed through
# WITHOUT normalization. MinMaxScaler is not applied to it because:
#   - it is already bounded to {0, 1} by construction (binary indicator)
#   - normalizing a binary column with MinMaxScaler fitted on continuous data
#     would be semantically meaningless and would not change its values
#   - downstream models (RF, AHP) treat it as a raw binary input
# All 12 continuous features are normalized to [0, 1] via MinMaxScaler.
features_normalized = features_raw[
    ['h3_id', 'label', 'cafe_count', 'viable_overlap_ratio',
     'pedestrian_street', 'geometry', 'centroid_x', 'centroid_y']
].copy()

for col in normalized_df.columns:
    features_normalized[col] = normalized_df[col].values

# Save scaler for reproducibility
scaler_path = os.path.join(GOLD_PATH, 'gold_minmax_scaler.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

# Verify: training cells should span full [0, 1] range; val cells may exceed it
norm_cols = [c for c in features_normalized.columns if c.endswith('_norm')]
train_check = features_normalized.iloc[train_idx][norm_cols]
val_check   = features_normalized.iloc[val_idx][norm_cols]
print("Normalization verification (training cells — should be exactly [0, 1]):")
print(train_check.agg(['min', 'max']).round(4).to_string())
print()
print("Normalization verification (validation cells — may slightly exceed [0, 1]):")
print(val_check.agg(['min', 'max']).round(4).to_string())

# Export
features_norm_wgs84 = features_normalized.to_crs(WGS84_CRS)
norm_path = os.path.join(GOLD_PATH, 'gold_features_normalized.geojson')
features_norm_wgs84.to_file(norm_path, driver='GeoJSON')
print(f"\nNormalized features saved: gold_features_normalized.geojson")
print(f"  Shape:              {features_normalized.shape}")
print(f"  Scaler:             fitted on {len(train_idx)} training cells only")
print(f"  sat_max_train:      {sat_max_train:.6f} (training cells only)")
print(f"  Artifacts saved:    gold_minmax_scaler.pkl, gold_sat_max_train.pkl")
print(f"  Split saved:        gold_train_val_idx.pkl")
# Clarification: 12 continuous features carry _norm suffix (MinMaxScaler applied);
# pedestrian_street is the 13th feature column, passed through as raw binary.
print(f"  Feature columns:    12 continuous _norm columns + 1 binary raw (pedestrian_street)")

Step 1 — Computing train/validation split (before normalization)...
  Train cells: 655  |  Validation cells: 316
  Split indices saved: gold_train_val_idx.pkl

Step 2 — Applying log1p transform to skewed features (on copy)...
  log1p applied to: ['metro_count', 'transit_density', 'office_density', 'retail_density', 'tourist_poi_count', 'poi_diversity', 'local_cafe_density', 'pop_density', 'night_light']
  Rationale for pop_density + night_light: right-skewed distributions with high outliers; log1p prevents outlier leverage in MinMaxScaler.
Step 3 — Inverting competitor_saturation (training-set max, no leakage)...
  sat_max_train = 1.000000 (saved to gold_sat_max_train.pkl)
Step 4 — Fitting MinMaxScaler on training cells only (no leakage)...
Normalization verification (training cells — should be exactly [0, 1]):
     metro_count_norm  transit_density_norm  network_centrality_norm  pop_density_norm  office_density_norm  university_proximity_norm  night_light_norm  retail_density_norm  to

# Cell 11

In [ ]:
log_lines = [
    f"Thesis — Phase 2 · Feature Engineering Log",
    f"============================================",
    f"Run date:       {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}",
    f"H3 cells:       {len(h3_viable)} viable cells",
    f"",
    f"Feature counts:",
    f"  15 GDF columns total   : 13 model features + viable_overlap_ratio + label",
    f"  13 RF model features   : all features passed to Random Forest",
    f"  10 AHP-scored features : RF features minus local_cafe_density, office_density, university_proximity (RF-only)",
    f"",
    f"Features computed (13 model features):",
    f"",
    f"  ACCESSIBILITY (AHP + RF)",
    f"    metro_count:          "
    f"mean={h3_viable['metro_count'].mean():.2f}, "
    f"max={h3_viable['metro_count'].max()}",
    f"    transit_density:      "
    f"mean={h3_viable['transit_density'].mean():.2f}, "
    f"max={h3_viable['transit_density'].max()}",
    f"    network_centrality:   "
    f"mean={h3_viable['network_centrality'].mean():.6f}",
    f"",
    f"  DEMAND POTENTIAL (AHP + RF)",
    f"    pop_density:          "
    f"mean={h3_viable['pop_density'].mean():.1f} res/km²",
    f"    office_density:       "
    f"mean={h3_viable['office_density'].mean():.2f}",
    f"    university_proximity: "
    f"mean={h3_viable['university_proximity'].mean():.4f} (exp-decay 0–1)",
    f"    night_light:          "
    f"mean={h3_viable['night_light'].mean():.4f}",
    f"",
    f"  URBAN CONTEXT (AHP + RF)",
    f"    retail_density:       "
    f"mean={h3_viable['retail_density'].mean():.2f}",
    f"    tourist_poi_count:    "
    f"mean={h3_viable['tourist_poi_count'].mean():.2f}",
    f"    poi_diversity:        "
    f"mean={h3_viable['poi_diversity'].mean():.2f}",
    f"    pedestrian_street:    "
    f"{h3_viable['pedestrian_street'].sum()} cells flagged",
    f"",
    f"  COMPETITION",
    f"    local_cafe_density:   "
    f"mean={h3_viable['local_cafe_density'].mean():.2f}  [RF-ONLY — excluded from AHP]",
    f"    competitor_saturation: "
    f"mean={h3_viable['competitor_saturation'].mean():.4f}  [AHP + RF, inverted before norm]",
    f"",
    f"  METADATA (not model features)",
    f"    viable_overlap_ratio: "
    f"mean={h3_viable['viable_overlap_ratio'].mean():.2f}",
    f"    label:                "
    f"{h3_viable['label'].sum()} positive, "
    f"{(h3_viable['label']==0).sum()} negative",
    f"",
    f"Normalization: MinMaxScaler [0,1] fitted on training cells only",
    f"  competitor_saturation inverted before normalization using sat_max_train",
    f"  (training-set max only — no leakage); stored as competitor_saturation_inv_norm",
    f"  sat_max_train value: {sat_max_train:.6f}",
    f"",
    f"Spatial split: GroupShuffleSplit on H3 resolution-7 groups",
    f"  Train cells: {len(train_idx)}, Validation cells: {len(val_idx)}",
    f"  Partial spatial separation — adjacent cells across group boundaries",
    f"  may still share correlated features within ~350 m (documented caveat)",
    f"",
    f"Population data source: ISTAT census tracts (block-level POP21)",
    f"University proximity: exponential decay exp(-d/1000m), range [0,1]",
    f"",
    f"Outputs:",
    f"  gold_features_raw.geojson       — 15 columns, untransformed",
    f"  gold_features_normalized.geojson — normalized features (12 continuous _norm columns + 1 binary raw: pedestrian_street)",
    f"  gold_minmax_scaler.pkl",
    f"  gold_sat_max_train.pkl",
    f"  gold_train_val_idx.pkl",
]

log_path = os.path.join(GOLD_PATH, 'gold_feature_log.txt')
with open(log_path, 'w') as f:
    f.write('\n'.join(log_lines))

print('\n'.join(log_lines))
print(f"\nLog saved to: {log_path}")

Thesis — Phase 2 · Feature Engineering Log
Run date:       2026-04-26 11:22
H3 cells:       971 viable cells

Feature counts:
  15 GDF columns total   : 13 model features + viable_overlap_ratio + label
  13 RF model features   : all features passed to Random Forest
  10 AHP-scored features : RF features minus local_cafe_density, office_density, university_proximity (RF-only)

Features computed (13 model features):

  ACCESSIBILITY (AHP + RF)
    metro_count:          mean=0.50, max=4
    transit_density:      mean=13.34, max=54
    network_centrality:   mean=0.001586

  DEMAND POTENTIAL (AHP + RF)
    pop_density:          mean=12377.8 res/km²
    office_density:       mean=17.85
    university_proximity: mean=0.3522 (exp-decay 0–1)
    night_light:          mean=53.0826

  URBAN CONTEXT (AHP + RF)
    retail_density:       mean=59.46
    tourist_poi_count:    mean=11.57
    poi_diversity:        mean=17.68
    pedestrian_street:    205 cells flagged

  COMPETITION
    local_cafe_densi

# Cell 12

In [ ]:
print("=" * 60)
print("NOTEBOOK 4 COMPLETE — Feature Engineering Summary")
print("=" * 60)

expected_files = [
    'gold_features_raw.geojson',
    'gold_features_normalized.geojson',
    'gold_minmax_scaler.pkl',
    'gold_sat_max_train.pkl',
    'gold_train_val_idx.pkl',
    'gold_feature_log.txt',
]

all_present = True
for filename in expected_files:
    filepath = os.path.join(GOLD_PATH, filename)
    status   = "OK" if os.path.exists(filepath) else "MISSING"
    if status == "MISSING":
        all_present = False
    print(f"  [{status}] {filename}")

print()
print(f"  GDF columns total:     15  (13 model features + viable_overlap_ratio + label)")
print(f"  RF model features:     13")
print(f"  AHP-scored features:   10  (local_cafe_density excluded — RF-only)")
print()
if all_present:
    print("All Gold outputs present.")
    print("Ready to proceed to Notebook 5 — AHP scoring.")
else:
    print("Some files missing. Review cells above before proceeding.")

NOTEBOOK 4 COMPLETE — Feature Engineering Summary
  [OK] gold_features_raw.geojson
  [OK] gold_features_normalized.geojson
  [OK] gold_minmax_scaler.pkl
  [OK] gold_sat_max_train.pkl
  [OK] gold_train_val_idx.pkl
  [OK] gold_feature_log.txt

  GDF columns total:     15  (13 model features + viable_overlap_ratio + label)
  RF model features:     13
  AHP-scored features:   10  (local_cafe_density excluded — RF-only)

All Gold outputs present.
Ready to proceed to Notebook 5 — AHP scoring.
